In [0]:
%sql
DROP TABLE IF EXISTS catalog_ventas.gold.kpi_ventas_hora;

CREATE OR REPLACE TABLE catalog_ventas.gold.kpi_ventas_hora
USING DELTA
AS

WITH base AS (
SELECT
    fa.id_venta,
    f.vtafecha,
    fa.total
FROM catalog_ventas.gold.fact_ventas fa

LEFT JOIN catalog_ventas.gold.dim_fecha f
  ON f.id_fecha = fa.id_fecha

)

SELECT
    DATE_FORMAT(vtafecha, 'HH:mm') AS hora,
    COUNT(DISTINCT id_venta) AS tickets,
    SUM(total)               AS facturacion
FROM base
GROUP BY hora
ORDER BY tickets DESC

In [0]:
%sql
SELECT * FROM catalog_ventas.gold.kpi_ventas_hora

In [0]:
%sql
/*
La mayor cantidad de tickets ocurre entre 15:00 y 17:00
La mayor facturación coincide con las mismas horas, aunque 17:00 es la más alta
Esto indica claramente que la tarde es el periodo pico de ventas
*/

In [0]:
%sql
SELECT *
FROM (
    SELECT
        DATE_FORMAT(vtafecha, 'HH:mm') AS hora,
        COUNT(DISTINCT id_venta)       AS tickets,
        SUM(total)                     AS facturacion,
        RANK() OVER (ORDER BY SUM(total) DESC) AS ranking_facturacion
    FROM base
    GROUP BY DATE_FORMAT(vtafecha, 'HH:mm')
) t
WHERE ranking_facturacion <= 10
ORDER BY ranking_facturacion;